In [1]:
from IPython.display import HTML
HTML('''<style>.jp-Cell-inputWrapper, .input { margin-top: 0.5em; }</style>''')

# Notebook 01 — Master DataFrame Construction
## ENGG2112 Project MODR

This notebook merges the per-source CSVs from Notebook 00 into the canonical **master cross-section** dataset, engineers rate-based features, and outputs the modelling-ready CSV used by every downstream notebook.

### Inputs
- `data/raw/usa_data/ny_flu_panel.csv` (1,032 rows, 17 seasons × 62 counties)
- `data/raw/usa_data/pennsylvania/pa_flu_panel.csv` (67 rows, 1 season × 67 counties)
- `data/raw/usa_data/connecticut/ct_flu_panel.csv` (27 rows, 3 seasons × 9 PRs)
- `data/raw/usa_data/delaware/de_flu_panel.csv` (18 rows, 6 seasons × 3 counties)
- `data/raw/usa_data/acs_demographics/acs_multistate_2022.csv` (141 areas)
- `data/raw/usa_data/acs_demographics/land_area_2022.csv` (141 areas)

### Outputs
- `data/processed/master_counties.csv` — primary cross-section (141 areas × ~25 columns)
- `data/processed/master_panel.csv` — full multi-season panel for sensitivity analysis (1,144 obs)
- `docs/data_dictionary.md` — auto-generated column documentation

### Pipeline steps
1. Load all sources
2. Build cross-section (most recent complete season per state)
3. Stack flu panels
4. Merge with ACS demographics + land area on FIPS
5. Engineer rate/percentage features
6. Compute outcome metrics (per-capita flu rate)
7. Compute outbreak label (within-state top 25%)
8. Quality checks
9. Build the panel version
10. Export both + auto-generate data dictionary

### Methodological choice — within-state outbreak labelling
We label "outbreak" as the **top 25% of areas within each state by per-capita flu rate**. This sidesteps the issue that different states have different testing rates and reporting practices, which would make absolute case counts incomparable across states. See `REFERENCES.md` for the rationale.

References for all data sources: see `REFERENCES.md` in the project root.

## Setup

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'usa_data'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
DOCS_DIR = PROJECT_ROOT / 'docs'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Raw dir:      {RAW_DIR}')
print(f'Processed:    {PROCESSED_DIR}')
print(f'Docs:         {DOCS_DIR}')

Project root: /Users/rocbarraket/Documents/ENGG2112/ENGG2112_Project
Raw dir:      /Users/rocbarraket/Documents/ENGG2112/ENGG2112_Project/data/raw/usa_data
Processed:    /Users/rocbarraket/Documents/ENGG2112/ENGG2112_Project/data/processed
Docs:         /Users/rocbarraket/Documents/ENGG2112/ENGG2112_Project/docs


---

# 1. Load Per-Source CSVs

We load each tidy CSV with `fips` as a string (preserves the leading zero for CT and DE).

In [3]:
# Force fips as string to preserve leading zeros
DTYPE = {'fips': str, 'state_fips': str}

ny = pd.read_csv(RAW_DIR / 'ny_flu_panel.csv', dtype=DTYPE)
pa = pd.read_csv(RAW_DIR / 'pennsylvania' / 'pa_flu_panel.csv', dtype=DTYPE)
ct = pd.read_csv(RAW_DIR / 'connecticut' / 'ct_flu_panel.csv', dtype=DTYPE)
de = pd.read_csv(RAW_DIR / 'delaware' / 'de_flu_panel.csv', dtype=DTYPE)
acs = pd.read_csv(RAW_DIR / 'acs_demographics' / 'acs_multistate_2022.csv', dtype=DTYPE)
land = pd.read_csv(RAW_DIR / 'acs_demographics' / 'land_area_2022.csv', dtype=DTYPE)

print(f'Loaded:')
print(f'  NY flu panel:     {ny.shape}')
print(f'  PA flu panel:     {pa.shape}')
print(f'  CT flu panel:     {ct.shape}')
print(f'  DE flu panel:     {de.shape}')
print(f'  ACS demographics: {acs.shape}')
print(f'  Land area:        {land.shape}')

Loaded:
  NY flu panel:     (1032, 12)
  PA flu panel:     (67, 13)
  CT flu panel:     (27, 12)
  DE flu panel:     (18, 12)
  ACS demographics: (141, 16)
  Land area:        (141, 3)


---

# 2. Season Selection for Cross-Section

We pick the most recent **complete or near-complete** season for each state. Where states differ in availability, we add `season_year` as a control variable.

| State | Selected season | Reason |
|---|---|---|
| **NY** | 2024-2025 | Most recent complete |
| **PA** | 2025-2026 | Only season available (in-progress, ~complete) |
| **CT** | 2024-2025 | Most recent complete |
| **DE** | 2024-2025 | Most recent complete |

Three out of four states are aligned at 2024-25. PA's 2025-26 in-progress data is essentially the same calendar window (Sept 2025 → April 2026) and most flu activity has completed by April. We add `season_start_year` as a feature for the model to control for any temporal effect.

In [4]:
# Pick season per state
SEASON_PER_STATE = {
    'NY': '2024-2025',
    'PA': '2025-2026',
    'CT': '2024-2025',
    'DE': '2024-2025',
}

ny_xs = ny[ny['season'] == SEASON_PER_STATE['NY']].copy()
pa_xs = pa[pa['season'] == SEASON_PER_STATE['PA']].copy()
ct_xs = ct[ct['season'] == SEASON_PER_STATE['CT']].copy()
de_xs = de[de['season'] == SEASON_PER_STATE['DE']].copy()

print('Selected per state:')
for state, df in [('NY', ny_xs), ('PA', pa_xs), ('CT', ct_xs), ('DE', de_xs)]:
    print(f'  {state} {SEASON_PER_STATE[state]}: {len(df)} areas')

print(f'\nTotal cross-section size: {len(ny_xs) + len(pa_xs) + len(ct_xs) + len(de_xs)} areas')

Selected per state:
  NY 2024-2025: 62 areas
  PA 2025-2026: 67 areas
  CT 2024-2025: 9 areas
  DE 2024-2025: 3 areas

Total cross-section size: 141 areas


---

# 3. Stack the Flu Cross-Section

We concatenate the four state-season slices into one DataFrame.

In [5]:
flu_xs = pd.concat([ny_xs, pa_xs, ct_xs, de_xs], ignore_index=True)
print(f'Stacked cross-section shape: {flu_xs.shape}')
print(f'Unique states: {flu_xs["state"].nunique()}')
print(f'Unique FIPS: {flu_xs["fips"].nunique()}')

# Sanity check — no duplicates
assert flu_xs['fips'].nunique() == len(flu_xs), "Duplicate FIPS in cross-section"

print(f'\nPer-state row counts:')
print(flu_xs['state'].value_counts())

print(f'\nMissing values per column:')
print(flu_xs.isnull().sum())

Stacked cross-section shape: (141, 13)
Unique states: 4
Unique FIPS: 141

Per-state row counts:
state
PA    67
NY    62
CT     9
DE     3
Name: count, dtype: int64

Missing values per column:
state                  0
state_fips             0
fips                   0
county                 0
season                 0
season_start_year      0
total_cases            0
weeks_active          67
peak_week_cases       67
time_to_peak          67
outbreak_steepness    67
pct_type_a            70
county.1              74
dtype: int64


---

# 4. Merge with ACS Demographics + Land Area

Every flu row should match exactly one ACS row and one land area row by FIPS.

In [6]:
# Pre-merge sanity check
flu_fips = set(flu_xs['fips'])
acs_fips = set(acs['fips'])
land_fips = set(land['fips'])

print(f'Flu FIPS:  {len(flu_fips)}')
print(f'ACS FIPS:  {len(acs_fips)}')
print(f'Land FIPS: {len(land_fips)}')
print(f'\nFlu missing from ACS:  {flu_fips - acs_fips}')
print(f'Flu missing from land: {flu_fips - land_fips}')

# Merge
master = flu_xs.merge(acs.drop(columns='state'), on='fips', how='left')
master = master.merge(land[['fips', 'land_area_sqmi']], on='fips', how='left')

print(f'\nAfter merge: {master.shape}')
assert len(master) == len(flu_xs), "Row count changed during merge"
assert master['pop_total'].notna().all(), "Some rows missing demographics"
assert master['land_area_sqmi'].notna().all(), "Some rows missing land area"
print('✅ Clean merge: no missing demographics or land area')

Flu FIPS:  141
ACS FIPS:  141
Land FIPS: 141

Flu missing from ACS:  set()
Flu missing from land: set()

After merge: (141, 28)
✅ Clean merge: no missing demographics or land area


---

# 5. Feature Engineering

Transform raw counts to **rates and percentages** so features are comparable across small (Hamilton, NY: 5K population) and large (Kings/Brooklyn: 2.6M population) areas.

### Rate-based features

| Engineered feature | Formula | Rationale |
|---|---|---|
| `pop_density_per_sqmi` | `pop_total / land_area_sqmi` | Density drives transmission rate |
| `pct_elderly` | `pop_elderly / pop_total × 100` | Elderly more vulnerable + higher healthcare contact |
| `poverty_rate` | `pop_below_poverty / pop_total × 100` | Socioeconomic access to care |
| `unemployment_rate` | `unemployed / labour_force × 100` | Standard BLS-style rate |
| `public_transport_pct` | `public_transport_commuters / total_commuters × 100` | High-contact commuting → exposure |
| `pct_bachelors_plus` | `pop_bachelors_plus / pop_total × 100` | Educational attainment proxy for SES |
| `pct_non_white` | `(pop_total - pop_white) / pop_total × 100` | Demographic diversity / minority share |
| `pct_foreign_born` | `pop_foreign_born / pop_total × 100` | International travel connectivity proxy |

In [7]:
# Engineered features — all rates and percentages
master['pop_density_per_sqmi'] = (master['pop_total'] / master['land_area_sqmi']).round(2)
master['pct_elderly'] = (master['pop_elderly'] / master['pop_total'] * 100).round(2)
master['poverty_rate'] = (master['pop_below_poverty'] / master['pop_total'] * 100).round(2)
master['unemployment_rate'] = (master['unemployed'] / master['labour_force'] * 100).round(2)
master['public_transport_pct'] = (master['public_transport_commuters'] / master['total_commuters'] * 100).round(2)
master['pct_bachelors_plus'] = (master['pop_bachelors_plus'] / master['pop_total'] * 100).round(2)
master['pct_non_white'] = ((master['pop_total'] - master['pop_white']) / master['pop_total'] * 100).round(2)
master['pct_foreign_born'] = (master['pop_foreign_born'] / master['pop_total'] * 100).round(2)

# These are the features we'll use for ML
FEATURE_COLS = [
    'pop_density_per_sqmi', 'median_age', 'pct_elderly',
    'avg_household_size',
    'median_income', 'poverty_rate', 'unemployment_rate',
    'public_transport_pct',
    'pct_bachelors_plus',
    'pct_non_white', 'pct_foreign_born'
]
print(f'Engineered {len(FEATURE_COLS)} features:')
for f in FEATURE_COLS:
    print(f'  - {f}')

print(f'\nFeature stats (across all 141 areas):')
print(master[FEATURE_COLS].describe().round(2).T)

Engineered 11 features:
  - pop_density_per_sqmi
  - median_age
  - pct_elderly
  - avg_household_size
  - median_income
  - poverty_rate
  - unemployment_rate
  - public_transport_pct
  - pct_bachelors_plus
  - pct_non_white
  - pct_foreign_born

Feature stats (across all 141 areas):
                      count      mean       std       min       25%       50%  \
pop_density_per_sqmi  141.0   1697.16   7697.64      2.96     69.60    143.30   
median_age            141.0     42.72      3.83     32.60     40.30     42.70   
pct_elderly           141.0     19.98      2.95     13.28     18.03     19.74   
avg_household_size    141.0      2.43      0.17      2.03      2.33      2.38   
median_income         141.0  71515.67  16174.35  46186.00  60557.00  67194.00   
poverty_rate          141.0     11.62      3.08      5.31      9.84     11.72   
unemployment_rate     141.0      5.29      1.25      2.18      4.51      5.25   
public_transport_pct  141.0      3.16      8.67      0.00      0.3

---

# 6. Outcome Metrics

Per-capita rates so areas of vastly different size are comparable.

| Outcome | Formula | Notes |
|---|---|---|
| `flu_rate_per_100k` | `total_cases / pop_total × 100,000` | Primary outcome — annual flu burden per 100K |
| `peak_rate_per_100k` | `peak_week_cases / pop_total × 100,000` | Outbreak intensity per 100K |

In [8]:
master['flu_rate_per_100k'] = (master['total_cases'] / master['pop_total'] * 100_000).round(1)
master['peak_rate_per_100k'] = (master['peak_week_cases'] / master['pop_total'] * 100_000).round(1)

print('Flu rate distribution by state:')
print(master.groupby('state')['flu_rate_per_100k'].describe().round(0))

Flu rate distribution by state:
       count    mean    std     min     25%     50%     75%     max
state                                                              
CT       9.0  1333.0  342.0   789.0  1144.0  1436.0  1549.0  1772.0
DE       3.0  2818.0  558.0  2303.0  2521.0  2740.0  3075.0  3410.0
NY      62.0  2250.0  796.0   925.0  1635.0  2150.0  2734.0  4332.0
PA      67.0  1178.0  572.0   216.0   797.0   972.0  1493.0  3006.0


---

# 7. Outbreak Label — Within-State Top 25%

We label "outbreak" as **top 25% of areas within each state by per-capita flu rate**. Why within-state and not pooled across states?

**Different states have systematically different testing rates, reporting practices, and lab capacities.** Pennsylvania's cumulative cases per 100K may not be directly comparable to New York's lab-confirmed cases per 100K. By labelling within state, we ask: *"Among the 62 NY counties, which had relatively worse flu burden? Among the 67 PA counties, which had relatively worse flu burden?"* — separately and meaningfully.

The model then learns: *"What demographic profile makes a county relatively vulnerable within its state's flu surveillance regime?"* — a defensible question that doesn't conflate state-level confounders with demographic effects.

In [9]:
def within_state_label(df, percentile=75):
    """Label top (100-percentile)% of areas per state as outbreak (1)."""
    out = df.copy()
    out['outbreak'] = 0
    for state in out['state'].unique():
        mask = out['state'] == state
        threshold = out.loc[mask, 'flu_rate_per_100k'].quantile(percentile / 100)
        out.loc[mask & (out['flu_rate_per_100k'] >= threshold), 'outbreak'] = 1
    return out

master = within_state_label(master, percentile=75)

print('Outbreak label distribution:')
print(master.groupby('state')['outbreak'].agg(['count', 'sum', 'mean']))
print(f'\nOverall outbreak rate: {master["outbreak"].mean():.1%}')
print(f'\nPer-state outbreak counties:')
for state in master['state'].unique():
    sub = master[(master['state'] == state) & (master['outbreak'] == 1)]
    counties = sub.sort_values('flu_rate_per_100k', ascending=False)['county'].tolist()
    print(f'  {state}: {counties}')

Outbreak label distribution:
       count  sum      mean
state                      
CT         9    3  0.333333
DE         3    1  0.333333
NY        62   16  0.258065
PA        67   17  0.253731

Overall outbreak rate: 26.2%

Per-state outbreak counties:
  NY: ['PUTNAM', 'WESTCHESTER', 'NASSAU', 'SUFFOLK', 'SULLIVAN', 'ROCKLAND', 'BRONX', 'ORANGE', 'WAYNE', 'CORTLAND', 'DUTCHESS', 'DELAWARE', 'QUEENS', 'RICHMOND', 'CHEMUNG', 'JEFFERSON']
  PA: ['WAYNE', 'CAMBRIA', 'POTTER', 'BRADFORD', 'MONROE', 'CARBON', 'ERIE', 'LEHIGH', 'NORTHAMPTON', 'TIOGA', 'MONTOUR', 'CUMBERLAND', 'SCHUYLKILL', 'BEDFORD', 'FULTON', 'HUNTINGDON', 'NORTHUMBERLAND']
  CT: ['GREATER BRIDGEPORT', 'NORTHEASTERN CONNECTICUT', 'SOUTHEASTERN CONNECTICUT']
  DE: ['KENT']


---

# 8. Quality Checks

Ensure the master dataset is clean and ready for modelling.

In [10]:
print('=== Quality Checks ===\n')

# 1. Row count
print(f'Total rows: {len(master)}')
assert len(master) == 141, f"Expected 141 rows, got {len(master)}"

# 2. No duplicate FIPS
assert master['fips'].is_unique, "Duplicate FIPS detected"
print('✅ No duplicate FIPS')

# 3. No missing values in key columns
key_cols = ['fips', 'state', 'county', 'pop_total', 'land_area_sqmi'] + FEATURE_COLS + ['flu_rate_per_100k', 'outbreak']
for col in key_cols:
    n_missing = master[col].isnull().sum()
    if n_missing > 0:
        print(f'⚠️  {col}: {n_missing} missing')
    else:
        print(f'✅ {col}: complete')

# 4. Sensible value ranges
print(f'\nSanity check on ranges:')
print(f'  Population: {master["pop_total"].min():,} to {master["pop_total"].max():,}')
print(f'  Density: {master["pop_density_per_sqmi"].min():.1f} to {master["pop_density_per_sqmi"].max():,.1f} per sq mi')
print(f'  Median age: {master["median_age"].min()} to {master["median_age"].max()}')
print(f'  Median income: ${master["median_income"].min():,} to ${master["median_income"].max():,}')
print(f'  Flu rate per 100K: {master["flu_rate_per_100k"].min():.1f} to {master["flu_rate_per_100k"].max():,.1f}')

=== Quality Checks ===

Total rows: 141
✅ No duplicate FIPS
✅ fips: complete
✅ state: complete
✅ county: complete
✅ pop_total: complete
✅ land_area_sqmi: complete
✅ pop_density_per_sqmi: complete
✅ median_age: complete
✅ pct_elderly: complete
✅ avg_household_size: complete
✅ median_income: complete
✅ poverty_rate: complete
✅ unemployment_rate: complete
✅ public_transport_pct: complete
✅ pct_bachelors_plus: complete
✅ pct_non_white: complete
✅ pct_foreign_born: complete
✅ flu_rate_per_100k: complete
✅ outbreak: complete

Sanity check on ranges:
  Population: 4,536 to 2,679,620
  Density: 3.0 to 72,639.6 per sq mi
  Median age: 32.6 to 56.4
  Median income: $46,186 to $137,709
  Flu rate per 100K: 215.5 to 4,332.2


---

# 9. Build the Panel Version (Sensitivity Analysis)

For the panel sensitivity analysis (one row per state-season-area), we stack ALL multi-season data from all states. This is used in Notebook 03 onwards as a robustness check on the cross-sectional findings.

### Panel composition
- NY: 1,032 obs (62 counties × 17 seasons, minus a few empty)
- PA: 67 obs (67 × 1)
- CT: 27 obs (9 × 3)
- DE: 18 obs (3 × 6)
- **Total: ~1,144 obs**

The same demographic features apply to every season for a given county (since we use a single ACS 2022 snapshot). The variation comes from per-season flu outcomes.

In [11]:
# Stack all multi-season data
panel_flu = pd.concat([ny, pa, ct, de], ignore_index=True)
print(f'Stacked panel flu rows: {len(panel_flu)}')

# Merge with demographics + land
panel = panel_flu.merge(acs.drop(columns='state'), on='fips', how='left')
panel = panel.merge(land[['fips', 'land_area_sqmi']], on='fips', how='left')

# Apply same feature engineering
panel['pop_density_per_sqmi'] = (panel['pop_total'] / panel['land_area_sqmi']).round(2)
panel['pct_elderly'] = (panel['pop_elderly'] / panel['pop_total'] * 100).round(2)
panel['poverty_rate'] = (panel['pop_below_poverty'] / panel['pop_total'] * 100).round(2)
panel['unemployment_rate'] = (panel['unemployed'] / panel['labour_force'] * 100).round(2)
panel['public_transport_pct'] = (panel['public_transport_commuters'] / panel['total_commuters'] * 100).round(2)
panel['pct_bachelors_plus'] = (panel['pop_bachelors_plus'] / panel['pop_total'] * 100).round(2)
panel['pct_non_white'] = ((panel['pop_total'] - panel['pop_white']) / panel['pop_total'] * 100).round(2)
panel['pct_foreign_born'] = (panel['pop_foreign_born'] / panel['pop_total'] * 100).round(2)

panel['flu_rate_per_100k'] = (panel['total_cases'] / panel['pop_total'] * 100_000).round(1)
panel['peak_rate_per_100k'] = (panel['peak_week_cases'] / panel['pop_total'] * 100_000).round(1)

# Within (state, season) outbreak label — top 25% per state per season
def within_state_season_label(df, percentile=75):
    out = df.copy()
    out['outbreak'] = 0
    for (state, season), grp in out.groupby(['state', 'season']):
        if len(grp) < 2:
            continue
        threshold = grp['flu_rate_per_100k'].quantile(percentile / 100)
        idx = grp.index[grp['flu_rate_per_100k'] >= threshold]
        out.loc[idx, 'outbreak'] = 1
    return out

panel = within_state_season_label(panel, percentile=75)

print(f'\nPanel shape: {panel.shape}')
print(f'Panel breakdown:')
print(panel.groupby('state')['season'].nunique().rename('seasons'))
print(f'\nPanel outbreak rate: {panel["outbreak"].mean():.1%}')

Stacked panel flu rows: 1144

Panel shape: (1144, 39)
Panel breakdown:
state
CT     3
DE     6
NY    17
PA     1
Name: seasons, dtype: int64

Panel outbreak rate: 26.0%


---

# 10. Export Master DataFrames

We save two CSVs:

| File | Purpose |
|---|---|
| `master_counties.csv` | Primary cross-section for Notebooks 02-08 |
| `master_panel.csv` | Multi-season panel for sensitivity / robustness checks |

In [12]:
# Cross-section export columns (in logical order)
XS_EXPORT_COLS = (
    # Identity
    ['state', 'state_fips', 'fips', 'county', 'NAME', 'season', 'season_start_year']
    # Outcome metrics
    + ['total_cases', 'flu_rate_per_100k', 'peak_week_cases', 'peak_rate_per_100k',
       'weeks_active', 'time_to_peak', 'outbreak_steepness', 'pct_type_a']
    # Engineered features (model inputs)
    + FEATURE_COLS
    # Raw demographics (kept for reference)
    + ['pop_total', 'pop_elderly', 'pop_below_poverty', 'labour_force', 'unemployed',
       'total_commuters', 'public_transport_commuters', 'pop_white', 'pop_foreign_born',
       'pop_bachelors_plus', 'land_area_sqmi']
    # Label
    + ['outbreak']
)

# Filter to columns that actually exist
xs_cols = [c for c in XS_EXPORT_COLS if c in master.columns]
master_export = master[xs_cols]
master_export.to_csv(PROCESSED_DIR / 'master_counties.csv', index=False)
print(f'✅ master_counties.csv: {master_export.shape}, {(PROCESSED_DIR / "master_counties.csv").stat().st_size / 1024:.1f} KB')

# Panel export
panel_cols = [c for c in XS_EXPORT_COLS if c in panel.columns]
panel_export = panel[panel_cols]
panel_export.to_csv(PROCESSED_DIR / 'master_panel.csv', index=False)
print(f'✅ master_panel.csv:    {panel_export.shape}, {(PROCESSED_DIR / "master_panel.csv").stat().st_size / 1024:.1f} KB')

print(f'\nExported to: {PROCESSED_DIR}/')

✅ master_counties.csv: (141, 38), 31.9 KB
✅ master_panel.csv:    (1144, 38), 261.5 KB

Exported to: /Users/rocbarraket/Documents/ENGG2112/ENGG2112_Project/data/processed/


---

# 11. Auto-Generate Data Dictionary

We produce a markdown data dictionary that documents every column in the master dataset — its source, formula, and dtype. This is committed alongside the data so future users can understand each variable.

In [13]:
# Build data dictionary
DATA_DICT = {
    # Identity / metadata
    'state': ('Identity', 'string', 'Two-letter state code (NY/PA/CT/DE)', 'Composed from state_fips'),
    'state_fips': ('Identity', 'string', '2-digit FIPS state code', 'US Census Bureau'),
    'fips': ('Identity', 'string', '5-digit FIPS county/area code (zero-padded)', 'US Census Bureau'),
    'county': ('Identity', 'string', 'County or planning region name (uppercase)', 'State DOH source'),
    'NAME': ('Identity', 'string', 'Full Census-formatted area name', 'US Census ACS'),
    'season': ('Identity', 'string', 'Flu season label (e.g. "2024-2025")', 'State DOH source'),
    'season_start_year': ('Identity', 'integer', 'Year the season started (for sorting)', 'Derived from season'),
    
    # Outcome metrics
    'total_cases': ('Outcome', 'integer', 'Total flu cases reported in season', 'State DOH'),
    'flu_rate_per_100k': ('Outcome', 'float', 'total_cases / pop_total × 100,000 — primary outcome', 'Engineered'),
    'peak_week_cases': ('Outcome', 'integer', 'Maximum single-week case count', 'State DOH (NY/CT/DE only)'),
    'peak_rate_per_100k': ('Outcome', 'float', 'peak_week_cases / pop_total × 100,000', 'Engineered'),
    'weeks_active': ('Outcome', 'integer', 'Number of weeks with case reports', 'State DOH (NY/CT/DE only)'),
    'time_to_peak': ('Outcome', 'integer', 'Weeks from season start to peak', 'Engineered (NY/CT/DE only)'),
    'outbreak_steepness': ('Outcome', 'float', 'peak / time_to_peak (cases per week to reach peak)', 'Engineered (NY/CT/DE only)'),
    'pct_type_a': ('Outcome', 'float', '% of cases that were Influenza A', 'State DOH (NY/CT only)'),

    # Engineered features (model inputs)
    'pop_density_per_sqmi': ('Feature', 'float', 'pop_total / land_area_sqmi', 'Engineered'),
    'median_age': ('Feature', 'float', 'Median age of population (years)', 'ACS B01002_001E'),
    'pct_elderly': ('Feature', 'float', 'pop_elderly / pop_total × 100', 'Engineered from ACS B01001'),
    'avg_household_size': ('Feature', 'float', 'Average household size', 'ACS B25010_001E'),
    'median_income': ('Feature', 'float', 'Median household income (USD)', 'ACS B19013_001E'),
    'poverty_rate': ('Feature', 'float', 'pop_below_poverty / pop_total × 100', 'Engineered from ACS B17001'),
    'unemployment_rate': ('Feature', 'float', 'unemployed / labour_force × 100', 'Engineered from ACS B23025'),
    'public_transport_pct': ('Feature', 'float', 'public_transport_commuters / total_commuters × 100', 'Engineered from ACS B08301'),
    'pct_bachelors_plus': ('Feature', 'float', 'pop_bachelors_plus / pop_total × 100', 'Engineered from ACS B15003'),
    'pct_non_white': ('Feature', 'float', '(pop_total - pop_white) / pop_total × 100', 'Engineered from ACS B02001'),
    'pct_foreign_born': ('Feature', 'float', 'pop_foreign_born / pop_total × 100', 'Engineered from ACS B05002'),
    
    # Raw demographics (reference)
    'pop_total': ('Raw demo', 'integer', 'Total population', 'ACS B01003_001E'),
    'pop_elderly': ('Raw demo', 'integer', 'Population aged 65+', 'ACS B01001 (sum 020-025, 044-049)'),
    'pop_below_poverty': ('Raw demo', 'integer', 'Population below poverty line', 'ACS B17001_002E'),
    'labour_force': ('Raw demo', 'integer', 'Civilian labour force aged 16+', 'ACS B23025_003E'),
    'unemployed': ('Raw demo', 'integer', 'Unemployed in labour force', 'ACS B23025_005E'),
    'total_commuters': ('Raw demo', 'integer', 'Total workers commuting to work', 'ACS B08301_001E'),
    'public_transport_commuters': ('Raw demo', 'integer', 'Commuters using public transport (bus/train/subway)', 'ACS B08301_010E'),
    'pop_white': ('Raw demo', 'integer', 'White-alone population', 'ACS B02001_002E'),
    'pop_foreign_born': ('Raw demo', 'integer', 'Foreign-born population', 'ACS B05002_013E'),
    'pop_bachelors_plus': ('Raw demo', 'integer', "Population with bachelor's degree or higher", 'ACS B15003 (sum 022-025)'),
    'land_area_sqmi': ('Raw demo', 'float', 'Land area in square miles', 'Census 2022 Gazetteer'),

    # Label
    'outbreak': ('Label', 'integer', 'Binary label: 1 = top 25% of areas in same state by flu_rate_per_100k', 'Engineered'),
}

# Write data dictionary
dd_path = DOCS_DIR / 'data_dictionary.md'
with open(dd_path, 'w') as f:
    f.write('# Data Dictionary — `master_counties.csv`\n\n')
    f.write('Auto-generated by Notebook 01. Documents every column in the master cross-section.\n\n')
    f.write(f'Cross-section size: {master_export.shape[0]} areas × {master_export.shape[1]} columns.\n\n')
    f.write('See `REFERENCES.md` for full data source attribution.\n\n')
    
    sections = ['Identity', 'Outcome', 'Feature', 'Raw demo', 'Label']
    for sect in sections:
        cols_in_sect = [(c, info) for c, info in DATA_DICT.items() if info[0] == sect]
        if not cols_in_sect:
            continue
        f.write(f'## {sect}\n\n')
        f.write('| Column | Type | Description | Source |\n')
        f.write('|---|---|---|---|\n')
        for col, (_, dtype, desc, source) in cols_in_sect:
            if col in master_export.columns:
                f.write(f'| `{col}` | {dtype} | {desc} | {source} |\n')
        f.write('\n')

    # Summary stats appendix
    f.write('## Summary Statistics — Cross-Section\n\n')
    f.write('```\n')
    summary_cols = ['flu_rate_per_100k'] + FEATURE_COLS
    f.write(master_export[summary_cols].describe().round(2).to_string())
    f.write('\n```\n\n')
    
    # State counts
    f.write('## State Composition\n\n')
    f.write('| State | Areas | Selected Season |\n')
    f.write('|---|---|---|\n')
    for state in ['NY', 'PA', 'CT', 'DE']:
        sub = master_export[master_export['state'] == state]
        n = len(sub)
        season = sub['season'].iloc[0] if n > 0 else '-'
        f.write(f'| {state} | {n} | {season} |\n')

print(f'✅ Data dictionary saved: {dd_path.relative_to(PROJECT_ROOT)}')
print(f'   Size: {dd_path.stat().st_size / 1024:.1f} KB')

✅ Data dictionary saved: docs/data_dictionary.md
   Size: 6.0 KB


---

## Summary

In [14]:
print('=== Master DataFrame Construction Complete ===\n')

print('📊 Cross-section dataset (master_counties.csv):')
print(f'   Shape: {master_export.shape}')
print(f'   Areas per state: {master_export["state"].value_counts().to_dict()}')
print(f'   Outbreak rate: {master_export["outbreak"].mean():.1%}')

print(f'\n📊 Panel dataset (master_panel.csv):')
print(f'   Shape: {panel_export.shape}')
print(f'   Total seasons covered: {panel_export["season"].nunique()}')

print(f'\n📁 Files written:')
for f in [PROCESSED_DIR / 'master_counties.csv', PROCESSED_DIR / 'master_panel.csv', DOCS_DIR / 'data_dictionary.md']:
    if f.exists():
        print(f'   ✅ {f.relative_to(PROJECT_ROOT)} ({f.stat().st_size / 1024:.1f} KB)')

print(f'\n👉 Next: open 02_eda.ipynb for exploratory data analysis on the master cross-section.')

=== Master DataFrame Construction Complete ===

📊 Cross-section dataset (master_counties.csv):
   Shape: (141, 38)
   Areas per state: {'PA': 67, 'NY': 62, 'CT': 9, 'DE': 3}
   Outbreak rate: 26.2%

📊 Panel dataset (master_panel.csv):
   Shape: (1144, 38)
   Total seasons covered: 17

📁 Files written:
   ✅ data/processed/master_counties.csv (31.9 KB)
   ✅ data/processed/master_panel.csv (261.5 KB)
   ✅ docs/data_dictionary.md (6.0 KB)

👉 Next: open 02_eda.ipynb for exploratory data analysis on the master cross-section.
